## 3661. Maximum Walls Destroyed by Robots (Hard)

There is an endless straight line populated with some robots and walls. You are given integer arrays robots, distance, and walls:

- robots[i] is the position of the ith robot.
- distance[i] is the maximum distance the ith robot's bullet can travel.
- walls[j] is the position of the jth wall.

Every robot has one bullet that can either fire to the left or the right at most distance[i] meters.

A bullet destroys every wall in its path that lies within its range. Robots are fixed obstacles: if a bullet hits another robot before reaching a wall, it immediately stops at that robot and cannot continue.

Return the maximum number of unique walls that can be destroyed by the robots.

Notes:

- A wall and a robot may share the same position; the wall can be destroyed by the robot at that position.
- Robots are not destroyed by bullets.

In [ ]:
import bisect

class Solution:
    def maxWalls(self, robots: list[int], distance: list[int], walls: list[int]) -> int:
        walls = sorted(walls)
        sorted_robots = sorted(zip(robots, distance))
        n = len(sorted_robots)
        if n == 0: return 0

        def count_wall(l, r):
            if l > r: return 0
            return bisect.bisect_right(walls, r) - bisect.bisect_left(walls, l)
        
        dp = [[0, 0] for _ in range(n)]

        # position, distance
        p0, d0 = sorted_robots[0]
        # 左邊可以射到幾個牆
        dp[0][0] = count_wall(p0 - d0, p0)
        r0_limit = min(p0 + d0, sorted_robots[1][0] if n > 1 else float('inf'))
        # 右邊可以射到幾個牆
        dp[0][1] = count_wall(p0, r0_limit)

        for i in range(1, n):
            curr_p, curr_d = sorted_robots[i]
            prev_p, prev_d = sorted_robots[i-1]
            next_p = sorted_robots[i+1][0] if i+1 < n else float('inf')
            
            # 當前機器人的左極限
            L_limit = max(curr_p - curr_d, prev_p)
            # 當前機器人的右極限
            R_limit = min(curr_p + curr_d, next_p)
            # 前一個機器人的右極限
            prev_R_reach = min(prev_p + prev_d, curr_p)

            # Case 1: 第i個機器人向左射
            # a_left為第i-1個機器人也向左射時的總牆數
            # 因為dp[i-1][0]已經涵蓋範圍在prev_p的牆了，故第i個機器人最遠只能射到prev_p + 1
            a_left = dp[i-1][0] + count_wall(max(L_limit, prev_p + 1), curr_p)
            # b_left為第i-1個機器人也向右射時的總牆數
            b_left = dp[i-1][1] + count_wall(max(L_limit, prev_R_reach + 1), curr_p)
            dp[i][0] = max(a_left, b_left)

            # Case 2: 第i個機器人向右射
            a_right = dp[i-1][0] + count_wall(curr_p, R_limit)
            b_right = dp[i-1][1] + count_wall(max(curr_p, prev_R_reach + 1), R_limit)
            dp[i][1] = max(a_right, b_right)

        return max(dp[-1])